In [0]:
from pyspark.sql import functions as F
from pyspark.sql.functions import trim, count, col

In [0]:
customer_bronze = spark.table("retail_catalog.bronze.customers")
product_bronze = spark.table("retail_catalog.bronze.products")
orders_bronze = spark.table("retail_catalog.bronze.orders")

customer_bronze.printSchema()
product_bronze.printSchema()
orders_bronze.printSchema()

### **Customer Silver Transformation**

In [0]:
silver_customers = (
    customer_bronze
    .select(
        F.trim("customer_id").alias("customer_id"),
        F.trim("first_name").alias("first_name"),
        F.trim("last_name").alias("last_name"),
        F.trim("gender").alias("gender"),
        F.lower(trim("email")).alias("email"),
        F.col("phone").cast("long").alias("phone"),
        F.to_date("date_of_birth",'dd-MM-yyyy').alias("date_of_birth"),
        F.trim("city").alias("city"),
        F.trim("state").alias("state"),
        F.trim("country").alias("country"),
        F.col("pincode").cast("integer").alias("pincode"),
        "ingestion_timestamp",
        "source_file",
        "batch_id"
    )
)

- trim       → removes accidental spaces
- lower      → standardizes email
- to_date    → converts string to DateType
- cast       → enforces the expected data type

In [0]:
#duplicate Customers
customer_duplicate = silver_customers.groupBy("customer_id").count().filter(F.col("count")>1)
display(customer_duplicate)

#Null Customers
Null_customer = silver_customers.filter(F.col("customer_id").isNull())
display(Null_customer)

#invali emails
invalid_email = silver_customers.filter(F.col("email").rlike(r"^[A-Za-z0-9._%+’-]+@[A-Za-z0-9.-]+\.[A-Za-z]{2,}$")== False)
display(invalid_email)

In [0]:
silver_customers.write.format('delta').mode('overwrite').saveAsTable('retail_catalog.silver.customers')

### **Product Transformation**

In [0]:
silver_products = (
    product_bronze
    .select(
        F.trim("product_id").alias("product_id"),
        F.trim("product_name").alias("product_name"),
        F.trim("category").alias("category"),
        F.trim("brand").alias("brand"),
        F.col("price").cast("double").alias("price"),
        F.col("stock_quantity").cast("integer").alias("stock_quantity"),
        F.trim("supplier").alias("supplier"),
        F.col("rating").cast("double").alias("rating"),
        F.to_date("launch_date", "yyyy-MM-dd").alias("launch_date"),
        F.trim("status").alias("status"),
        "ingestion_timestamp",
        "source_file",
        "batch_id"
    )
)

display(silver_products)

In [0]:
invalid_products = silver_products.filter(
    (F.col("price") <= 0) |
    (F.col("stock_quantity") < 0) |
    (F.col("rating") < 0) |
    (F.col("rating") > 5)
)

display(invalid_products)

#Duplicate Product ID
product_duplicates = (
    silver_products
    .groupBy("product_id")
    .count()
    .filter(F.col("count") > 1)
)

display(product_duplicates)

In [0]:
(
    silver_products
    .write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(
        "retail_catalog.silver.products"
    )
)

### **Order Transformation**

In [0]:
silver_orders = (
    orders_bronze
    .select(
        F.trim("order_id").alias("order_id"),
        F.trim("customer_id").alias("customer_id"),
        F.trim("product_id").alias("product_id"),
        F.col("quantity").cast("integer").alias("quantity"),
        F.col("unit_price").cast("double").alias("unit_price"),
        F.col("discount").cast("integer").alias("discount"),
        F.col("total_amount").cast("double").alias("total_amount"),
        F.trim("payment_method").alias("payment_method"),
        F.trim("order_status").alias("order_status"),
        F.to_date("order_date", "yyyy-MM-dd").alias("order_date"),
        "ingestion_timestamp",
        "source_file",
        "batch_id"
    )
)

display(silver_orders)

In [0]:
# Invalid Quantity
invalid_quantity = silver_orders.filter(F.col("quantity")<=0)
display(invalid_quantity)

# Invalid Discount
invalid_discount = silver_orders.filter((F.col("discount")<0) | (F.col("discount")>100))
display(invalid_discount)

# Invalid Unit Price
invalid_unit_price = silver_orders.filter(F.col("unit_price") <= 0)
display(invalid_unit_price)

